In [ ]:
import sys
!{sys.executable} -m pip install crewai python-dotenv litellm nest_asyncio


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.5/89.5 kB 7.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 7.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.0/71.0 kB 6.0 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of opentelemetry-exporter-otlp-proto-grpc to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 23.1 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.9/185.9 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 87.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━

In [3]:
import requests

FIREWORKS_API_KEY = "fw_5ye6tWo8YJWqzXgRkpAGoT"

resp = requests.get(
    "https://api.fireworks.ai/inference/v1/models",
    headers={"Authorization": f"Bearer {FIREWORKS_API_KEY}"}
)
print(resp.status_code)
for m in resp.json().get("data", []):
    print(m["id"])

200
accounts/fireworks/models/flux-1-schnell-fp8
accounts/fireworks/models/gpt-oss-120b
accounts/fireworks/models/glm-5p1
accounts/fireworks/models/deepseek-v4-pro
accounts/fireworks/models/kimi-k2p6
accounts/fireworks/models/kimi-k2p5
accounts/fireworks/models/glm-5p2


In [4]:
import os
import nest_asyncio
import litellm
from crewai import Agent, Task, Crew, LLM

nest_asyncio.apply()

# ================== FORCE CLEAN FIREWORKS CALLS ==================
FIREWORKS_API_KEY = "fw_5ye6tWo8YJWqzXgRkpAGoT"

# Custom completion wrapper to strip problematic params
original_completion = litellm.completion

def clean_fireworks_completion(*args, **kwargs):
    # Remove any cache-related or extra fields Fireworks doesn't like
    for msg in kwargs.get("messages", []):
        if isinstance(msg, dict):
            msg.pop("cache_breakpoint", None)
    
    # Clean kwargs
    kwargs.pop("cache_breakpoint", None)
    if "extra_body" in kwargs:
        kwargs["extra_body"] = {k: v for k, v in kwargs["extra_body"].items() 
                              if k not in ["cache_breakpoint"]}
    
    return original_completion(*args, **kwargs)

litellm.completion = clean_fireworks_completion

# ================== LLM CONFIG ==================
llm = LLM(
    model="fireworks_ai/accounts/fireworks/models/deepseek-v4-pro",
    base_url="https://api.fireworks.ai/inference/v1",
    api_key=FIREWORKS_API_KEY,
    temperature=0.3,
    max_tokens=2048,
    top_p=0.9,
)

# ================== TEST SIMPLE CREW ==================
test_agent = Agent(
    role="Test Agent",
    goal="Answer factual questions accurately",
    backstory="You are a reliable AI assistant.",
    llm=llm,
    verbose=True,
    allow_delegation=False
)

test_task = Task(
    description="What is the capital of Algeria?",
    expected_output="Just the name of the capital city.",
    agent=test_agent
)

crew = Crew(
    agents=[test_agent],
    tasks=[test_task],
    verbose=True
)

result = await crew.kickoff_async()
print("\n=== RESULT ===")
print(result)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 28020516-726f-49d7-aad3-78f20ce44d21                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: What is the capital of Algeria?                                                                          │
│  ID: 31b99762-d106-4c30-885f-0f83e3a0e330                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Test Agent                                                                                              │
│                                                                                                                 │
│  Task: What is the capital of Algeria?                                                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Test Agent                                                                                              │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Algiers                                                                                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: What is the capital of Algeria?                                                                          │
│  Agent: Test Agent                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 28020516-726f-49d7-aad3-78f20ce44d21                                                                       │
│  Final Output: Algiers                                                                                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


=== RESULT ===
Algiers




╭────────────────────────── Tracing Preference Saved ──────────────────────────╮
│                                                                              │
│  Info: Tracing has been disabled.                                            │
│                                                                              │
│  Your preference has been saved. Future Crew/Flow executions will not        │
│  collect traces.                                                             │
│                                                                              │
│  To enable tracing later, do any one of these:                               │
│  • Set tracing=True in your Crew/Flow code                                   │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file               │
│  • Run: crewai traces enable                                                 │
│                                                                              │
╰─────────────────────────

In [5]:
import sys
!{sys.executable} -m pip install crewai_tools


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 811.5/811.5 kB 15.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 70.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.2/485.2 kB 30.2 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: tiktoken
    Found existing installation: tiktoken 0.13.0
    Uninstalling tiktoken-0.13.0:
      Successfully uninstalled tiktoken-0.13.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are ins

In [ ]:
import os
import nest_asyncio
import litellm
from crewai import Agent, Task, Crew, LLM

# We will avoid GithubSearchTool for now to prevent OpenAI dependency
# and use a simpler, more reliable approach

nest_asyncio.apply()

# ================== FIREWORKS CLEAN PATCH ==================
FIREWORKS_API_KEY = "fw_5ye6tWo8YJWqzXgRkpAGoT"

original_completion = litellm.completion

def clean_fireworks_completion(*args, **kwargs):
    for msg in kwargs.get("messages", []):
        if isinstance(msg, dict):
            msg.pop("cache_breakpoint", None)
    kwargs.pop("cache_breakpoint", None)
    if "extra_body" in kwargs:
        kwargs["extra_body"] = {k: v for k, v in kwargs.get("extra_body", {}).items() 
                              if k not in ["cache_breakpoint"]}
    return original_completion(*args, **kwargs)

litellm.completion = clean_fireworks_completion

# ================== LLM ==================
llm = LLM(
    model="fireworks_ai/accounts/fireworks/models/gpt-oss-120b",
    base_url="https://api.fireworks.ai/inference/v1",
    api_key=FIREWORKS_API_KEY,
    temperature=0.65,
    max_tokens=2048,
)

# ================== AGENTS (No heavy tools) ==================
repo_analyzer = Agent(
    role="GitHub Repository Analyst",
    goal="Understand the project and its value proposition",
    backstory="You are an expert at analyzing GitHub repositories from their README and description.",
    llm=llm,
    verbose=True
)

marketing_writer = Agent(
    role="Technical Marketing Copywriter",
    goal="Write engaging promotional content for GitHub projects",
    backstory="Expert in creating viral launch posts for developers.",
    llm=llm,
    verbose=True
)

growth_expert = Agent(
    role="Open Source Growth Strategist",
    goal="Create effective launch and community strategies",
    backstory="Specialist in hackathon and GitHub project growth.",
    llm=llm,
    verbose=True
)

# ================== TASKS ==================
task1 = Task(
    description="""Analyze this GitHub repository: {repo_url}
    Understand what the project does, its main features, target users, and unique strengths.""",
    expected_output="Clear project summary and positioning.",
    agent=repo_analyzer
)

task2 = Task(
    description="Write 3 high-quality promotional posts based on the analysis.",
    expected_output="Ready-to-post content for LinkedIn/X, Reddit, and Hacker News.",
    agent=marketing_writer
)

task3 = Task(
    description="Create a practical growth and launch strategy for this repository.",
    expected_output="Actionable plan with channels and timeline.",
    agent=growth_expert
)

# ================== CREW ==================
crew = Crew(
    agents=[repo_analyzer, marketing_writer, growth_expert],
    tasks=[task1, task2, task3],
    verbose=True
)

# ================== RUN (Manual Input) ==================
print("📥 GitHub Marketing Strategy Agent Ready!\n")

repo_url = input("Paste GitHub Repository URL: ").strip()

if not repo_url:
    repo_url = "https://github.com/WafaQueen1/KeepBeat"

print(f"\n🔍 Analyzing → {repo_url}\n")

result = crew.kickoff(inputs={"repo_url": repo_url})

print("\n" + "="*70)
print("🏆 MARKETING STRATEGY GENERATED SUCCESSFULLY!")
print("="*70)
print(result)

📥 GitHub Marketing Strategy Agent Ready!



In [1]:
import os
import nest_asyncio
import litellm
from crewai import Agent, Task, Crew, LLM

nest_asyncio.apply()

# ================== FIREWORKS PATCH ==================
FIREWORKS_API_KEY = "fw_5ye6tWo8YJWqzXgRkpAGoT"

original_completion = litellm.completion

def clean_fireworks(*args, **kwargs):
    for msg in kwargs.get("messages", []):
        if isinstance(msg, dict):
            msg.pop("cache_breakpoint", None)
    kwargs.pop("cache_breakpoint", None)
    if "extra_body" in kwargs:
        kwargs["extra_body"] = {k: v for k, v in kwargs.get("extra_body", {}).items() 
                              if k not in ["cache_breakpoint"]}
    return original_completion(*args, **kwargs)

litellm.completion = clean_fireworks

# ================== LLM ==================
llm = LLM(
    model="fireworks_ai/accounts/fireworks/models/gpt-oss-120b",
    base_url="https://api.fireworks.ai/inference/v1",
    api_key=FIREWORKS_API_KEY,
    temperature=0.65,
    max_tokens=2048,
)

# ================== AGENTS ==================
repo_analyzer = Agent(
    role="GitHub Repository Analyst",
    goal="Understand the project deeply",
    backstory="Expert at analyzing GitHub repos from description and README.",
    llm=llm,
    verbose=True
)

marketing_writer = Agent(
    role="Technical Marketing Copywriter",
    goal="Create compelling promotional content",
    backstory="Skilled writer of developer-focused launch posts.",
    llm=llm,
    verbose=True
)

growth_expert = Agent(
    role="Growth Strategist",
    goal="Build effective launch strategies",
    backstory="Expert in open source and hackathon project growth.",
    llm=llm,
    verbose=True
)

# ================== TASKS ==================
task1 = Task(
    description="Analyze repository: {repo_url}. Summarize what it does, key features, and target users.",
    expected_output="Project summary and positioning.",
    agent=repo_analyzer
)

task2 = Task(
    description="Write 3 promotional posts (LinkedIn/X, Reddit, Hacker News style).",
    expected_output="Ready-to-post markdown content.",
    agent=marketing_writer
)

task3 = Task(
    description="Create a practical growth & launch plan for this repo.",
    expected_output="Actionable strategy.",
    agent=growth_expert
)

# ================== CREW ==================
crew = Crew(
    agents=[repo_analyzer, marketing_writer, growth_expert],
    tasks=[task1, task2, task3],
    verbose=True
)

# ================== RUN ==================
print("📥 GitHub Marketing Agent Ready!\n")

repo_url = input("Paste GitHub Repository URL: ").strip()
if not repo_url:
    repo_url = "https://github.com/WafaQueen1/KeepBeat"

print(f"\n🔍 Analyzing → {repo_url}\n")

# Use async version inside Jupyter
result = await crew.kickoff_async(inputs={"repo_url": repo_url})

print("\n" + "="*70)
print("🏆 SUCCESS! MARKETING STRATEGY GENERATED")
print("="*70)
print(result)

📥 GitHub Marketing Agent Ready!


🔍 Analyzing → https://github.com/WafaQueen1/KeepBeat



╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 8a27235b-2f16-498c-b315-5cd10647fd86                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Analyze repository: https://github.com/WafaQueen1/KeepBeat. Summarize what it does, key features, and    │
│  target users.                                                                                                  │
│  ID: 3241e778-8ade-4e99-9620-adf1fb5884eb                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: GitHub Repository Analyst                                                                               │
│                                                                                                                 │
│  Task: Analyze repository: https://github.com/WafaQueen1/KeepBeat. Summarize what it does, key features, and    │
│  target users.                                                                                                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Growth Strategist                                                                                       │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # KeepBeat – Growth & Launch Playbook                                                                          │
│  **Goal:** Turn KeepBeat (open‑source adaptive metronome + practice logger) into a vibrant, self‑sustaining     │
│  community that drives downloads, contributions, and long‑term adoption across musicians, educators, and        │
│  developers.                                                                                                    │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## 1️⃣  Project Snapshot (for reference)                                                                        │
│                                                                                                                 │
│  | **What** | Adaptive, Rust + WASM metronome + practice logger (offline‑first, optional Git sync). |           │
│  |----------|--------------------------------------------------------------------------------------|            │
│  | **Core Benefits** | • Real‑time audio analysis & tempo feedback  <br>• Swing / polyrhythm / custom           │
│  signatures  <br>• Practice stats & streaks  <br>• Tiny, zero‑dependency binaries (≤ 5 MB)  <br>•               │
│  Cross‑platform (Win/macOS/Linux + browser) |                                                                   │
│  | **License** | MIT (commercial‑friendly) |                                                                    │
│  | **Primary Audiences** | • Instrumentalists & vocalists  <br>• Music teachers & schools  <br>• DIY            │
│  audio‑devs & Rust enthusiasts  <br>• Indie producers / DAW users |                                             │
│  | **Key Differentiators** | • “Listening” metronome (audio‑input analysis)  <br>• Git‑backed JSON sync (no     │
│  lock‑in cloud)  <br>• 100 % Rust → performance & safety  <br>• Fully open‑source, easy to fork or embed |      │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## 2️⃣  Success Metrics (KPIs)                                                                                  │
│                                                                                                                 │
│  | Metric | Target (12 mo) | Measurement Tool |                                                                 │
│  |--------|----------------|------------------|                                                                 │
│  | **GitHub Stars** | 2 000 | GitHub Insights |                                                                 │
│  | **Forks** | 600 | GitHub Insights |                                                                          │
│  | **Active Contributors** | 30 (≥ 5 PR/month) | GitHub PR stats |                                              │
│  | **Monthly Active Users (MAU)** | 5 000 (desktop + 

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Create a practical growth & launch plan for this repo.                                                   │
│  Agent: Growth Strategist                                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


🏆 SUCCESS! MARKETING STRATEGY GENERATED
# KeepBeat – Growth & Launch Playbook  
**Goal:** Turn KeepBeat (open‑source adaptive metronome + practice logger) into a vibrant, self‑sustaining community that drives downloads, contributions, and long‑term adoption across musicians, educators, and developers.  

---

## 1️⃣  Project Snapshot (for reference)

| **What** | Adaptive, Rust + WASM metronome + practice logger (offline‑first, optional Git sync). |
|----------|--------------------------------------------------------------------------------------|
| **Core Benefits** | • Real‑time audio analysis & tempo feedback  <br>• Swing / polyrhythm / custom signatures  <br>• Practice stats & streaks  <br>• Tiny, zero‑dependency binaries (≤ 5 MB)  <br>• Cross‑platform (Win/macOS/Linux + browser) |
| **License** | MIT (commercial‑friendly) |
| **Primary Audiences** | • Instrumentalists & vocalists  <br>• Music teachers & schools  <br>• DIY audio‑devs & Rust enthusiasts  <br>• Indie producers / DAW

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 8a27235b-2f16-498c-b315-5cd10647fd86                                                                       │
│  Final Output: # KeepBeat – Growth & Launch Playbook                                                            │
│  **Goal:** Turn KeepBeat (open‑source adaptive metronome + practice logger) into a vibrant, self‑sustaining     │
│  community that drives downloads, contributions, and long‑term adoption across musicians, educators, and        │
│  developers.                                                                                                    │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## 1️⃣  Project Snapshot (for reference)                                                                        │
│                                                                                                                 │
│  | **What** | Adaptive, Rust + WASM metronome + practice logger (offline‑first, optional Git sync). |           │
│  |----------|--------------------------------------------------------------------------------------|            │
│  | **Core Benefits** | • Real‑time audio analysis & tempo feedback  <br>• Swing / polyrhythm / custom           │
│  signatures  <br>• Practice stats & streaks  <br>• Tiny, zero‑dependency binaries (≤ 5 MB)  <br>•               │
│  Cross‑platform (Win/macOS/Linux + browser) |                                                                   │
│  | **License** | MIT (commercial‑friendly) |                                                                    │
│  | **Primary Audiences** | • Instrumentalists & vocalists  <br>• Music teachers & schools  <br>• DIY            │
│  audio‑devs & Rust enthusiasts  <br>• Indie producers / DAW users |                                             │
│  | **Key Differentiators** | • “Listening” metronome (audio‑input analysis)  <br>• Git‑backed JSON sync (no     │
│  lock‑in cloud)  <br>• 100 % Rust → performance & safety  <br>• Fully open‑source, easy to fork or embed |      │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## 2️⃣  Success Metrics (KPIs)                                                                                  │
│                                                                                                                 │
│  | Metric | Target (12 mo) | Measurement Tool |                                                                 │
│  |--------|----------------|------------------|                                                                 │
│  | **GitHub Stars** | 2 000 | GitHub Insights |                                                                 │
│  | **Forks** | 600 | GitHub Insights |                                                                          │
│  | **Active Contributors** | 30 (≥ 5 PR/month) | GitHub PR stats |                                              │
│  | **Monthly Active Users (MAU)** | 5 000 (desktop +

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [2]:
import litellm

response = litellm.completion(
    model="fireworks_ai/accounts/fireworks/models/gpt-oss-120b",
    messages=[{"role": "user", "content": "Say hello in one word"}],
    api_key="fw_5ye6tWo8YJWqzXgRkpAGoT",
    base_url="https://api.fireworks.ai/inference/v1"
)

print("Response:", response.choices[0].message.content)
print("Tokens used:", response.usage.total_tokens)

Response: Hello
Tokens used: 147


In [3]:
import nest_asyncio
import requests
from crewai import Agent, Task, Crew, LLM

nest_asyncio.apply()

FIREWORKS_API_KEY = "fw_5ye6tWo8YJWqzXgRkpAGoT"

# ================== README READER ==================
def read_github_readme(repo_url: str) -> str:
    try:
        for branch in ["main", "master"]:
            raw_url = repo_url.replace("github.com", "raw.githubusercontent.com") + f"/{branch}/README.md"
            r = requests.get(raw_url, timeout=15)
            if r.status_code == 200:
                print("✅ README fetched successfully")
                return r.text[:13000]
    except Exception as e:
        print(f"⚠️ README fetch failed: {e}")
    return "Could not fetch README content."

# ================== LLM (No patch) ==================
llm = LLM(
    model="fireworks_ai/accounts/fireworks/models/gpt-oss-120b",
    base_url="https://api.fireworks.ai/inference/v1",
    api_key=FIREWORKS_API_KEY,
    temperature=0.65,
    max_tokens=2048,
)

# ================== AGENTS ==================
repo_analyzer = Agent(
    role="GitHub Repository Analyst",
    goal="Analyze the repository and README",
    backstory="You are an expert at understanding open source projects.",
    llm=llm,
    verbose=True
)

marketing_writer = Agent(
    role="Technical Marketing Copywriter",
    goal="Write engaging promotional posts",
    backstory="Expert in creating content for developers on LinkedIn, X, Reddit, and HN.",
    llm=llm,
    verbose=True
)

growth_expert = Agent(
    role="Open Source Growth Strategist",
    goal="Create effective launch and growth strategy",
    backstory="Specialist in growing GitHub repositories.",
    llm=llm,
    verbose=True
)

# ================== TASKS ==================
task1 = Task(
    description="""Repository URL: {repo_url}

README:
{readme_content}

Analyze and summarize: purpose, key features, target users, and unique strengths.""",
    expected_output="Clear project summary",
    agent=repo_analyzer
)

task2 = Task(
    description="Write 3 ready-to-post promotional messages (LinkedIn/X, Reddit, Hacker News).",
    expected_output="High-quality marketing posts",
    agent=marketing_writer
)

task3 = Task(
    description="Create a practical growth and launch strategy for this repository.",
    expected_output="Actionable plan with steps and timeline.",
    agent=growth_expert
)

crew = Crew(
    agents=[repo_analyzer, marketing_writer, growth_expert],
    tasks=[task1, task2, task3],
    verbose=True
)

# ================== RUN ==================
print("📥 Agent Ready!\n")
repo_url = input("Paste GitHub URL: ").strip() or "https://github.com/WafaQueen1/my-microservices-aws-devops"

print(f"\n🔍 Analyzing → {repo_url}\n")
readme_content = read_github_readme(repo_url)

result = await crew.kickoff_async(inputs={
    "repo_url": repo_url,
    "readme_content": readme_content
})

print("\n" + "="*85)
print("🏆 MARKETING STRATEGY READY!")
print("="*85)
print(result)

📥 Agent Ready!


🔍 Analyzing → https://github.com/WafaQueen1/my-microservices-aws-devops

✅ README fetched successfully


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 99b78403-fe5f-4adc-8c3f-fb44a2a54887                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Repository URL: https://github.com/WafaQueen1/my-microservices-aws-devops                                │
│                                                                                                                 │
│  README:                                                                                                        │
│  # 🛒 E-Commerce Microservices Architecture on AWS EKS                                                          │
│                                                                                                                 │
│  This project demonstrates a production-grade, highly available microservices application deployed on **AWS     │
│  Elastic Kubernetes Service (EKS)**. It features a complete CI/CD pipeline, automated infrastructure            │
│  provisioning, and a robust monitoring stack (Prometheus & Grafana) along with AWS CloudWatch integration.      │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## 🏛️ Project Architecture                                                                                     │
│                                                                                                                 │
│  ![Project Architecture](FINAL_architecture_diagram.png)                                                        │
│                                                                                                                 │
│  ### 📊 System Workflow                                                                                         │
│                                                                                                                 │
│  The architecture follows a standard 3-tier model adapted for Kubernetes:                                       │
│                                                                                                                 │
│  1.  **Ingress Layer**: AWS Classic Load Balancers routing traffic to the EKS nodes.                            │
│  2.  **Logic Layer**: Microservices running in the `ecommerce` namespace with 2 replicas each for High          │
│  Availability.                                                                                                  │
│  3.  **Data Layer**: AWS RDS MySQL instance located in isolated private subnets.                                │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## 🏗️ Technical Component Breakdown                                                                            │
│                                                                                                                 │
│  ### 1. AWS Infrastructure (Terraform)                                                                          │
│                                                                                                                 │
│  The environment is provisioned using highly modular Terraform code:                                            │
│                                                          

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: GitHub Repository Analyst                                                                               │
│                                                                                                                 │
│  Task: Repository URL: https://github.com/WafaQueen1/my-microservices-aws-devops                                │
│                                                                                                                 │
│  README:                                                                                                        │
│  # 🛒 E-Commerce Microservices Architecture on AWS EKS                                                          │
│                                                                                                                 │
│  This project demonstrates a production-grade, highly available microservices application deployed on **AWS     │
│  Elastic Kubernetes Service (EKS)**. It features a complete CI/CD pipeline, automated infrastructure            │
│  provisioning, and a robust monitoring stack (Prometheus & Grafana) along with AWS CloudWatch integration.      │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## 🏛️ Project Architecture                                                                                     │
│                                                                                                                 │
│  ![Project Architecture](FINAL_architecture_diagram.png)                                                        │
│                                                                                                                 │
│  ### 📊 System Workflow                                                                                         │
│                                                                                                                 │
│  The architecture follows a standard 3-tier model adapted for Kubernetes:                                       │
│                                                                                                                 │
│  1.  **Ingress Layer**: AWS Classic Load Balancers routing traffic to the EKS nodes.                            │
│  2.  **Logic Layer**: Microservices running in the `ecommerce` namespace with 2 replicas each for High          │
│  Availability.                                                                                                  │
│  3.  **Data Layer**: AWS RDS MySQL instance located in isolated private subnets.                                │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## 🏗️ Technical Component Breakdown                                                                            │
│                                                                                                                 │
│  ### 1. AWS Infrastructure (Terraform)                                                                          │
│                                                                                                                 │
│  The environment is provisioned using highly modular Terr

[Finalize] todos_count=0, todos_with_results=0


...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Open Source Growth Strategist                                                                           │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # 📈 Growth & Launch Strategy for **“E‑Commerce Microservices Architecture on AWS EKS”**                       │
│  *(GitHub repo: https://github.com/yourorg/ecommerce-eks)*                                                      │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## 1️⃣ Strategy Overview                                                                                        │
│                                                                                                                 │
│  | Objective | Success Metric (30‑day) | Success Metric (90‑day) |                                              │
│  |-----------|------------------------|--------------------------|                                              │
│  | **Awareness** | 2 000 unique visitors to the repo page, 5 000 impressions on social posts | 10 000 unique    │
│  visitors, 25 000 impressions |                                                                                 │
│  | **Community** | 30 star ⭐, 5 forks, 2 external PRs | 100 star, 25 forks, 10 external PRs |                  │
│  | **Adoption** | 10 fresh‑clone “quick‑start” runs (tracked via GitHub Actions badge) | 30 runs, 5             │
│  organizations reporting production use (via issue template) |                                                  │
│  | **Lead Generation** | 20 newsletter sign‑ups / contact forms from the repo landing page | 50 sign‑ups, 3     │
│  demo‑request meetings |                                                                                        │
│  | **Revenue (optional)** | 1 paid consulting/implementation inquiry | 3 inquiries, $5k+ pipeline |             │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## 2️⃣ Target Audiences (recapped)                                                                              │
│                                                                                                                 │
│  | Segment | Primary Pain Point | How the Repo Solves It |                                                      │
│  |---------|-------------------|------------------------|                                                       │
│  | **DevOps engineers / Platform teams** | Need a production‑grade, end‑to‑end blueprint (IaC + CI/CD +         │
│  Observability) | Full Terraform + Jenkins + Prometheus stack, ready to fork |                                  │
│  | **Students & AWS Academy participants** | Limited IAM permissions, cost‑concerns | Works with `LabRole`,     │
│  includes cost warnings & destroy script |                                                                      │
│  | **Software architects** | Want a reference for micro‑service design on EKS | HA‑ready architecture, RBAC,    │
│  multi‑pod Kaniko |                                    

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Create a practical growth and launch strategy for this repository.                                       │
│  Agent: Open Source Growth Strategist                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


🏆 MARKETING STRATEGY READY!
# 📈 Growth & Launch Strategy for **“E‑Commerce Microservices Architecture on AWS EKS”**  
*(GitHub repo: https://github.com/yourorg/ecommerce-eks)*  

---

## 1️⃣ Strategy Overview  

| Objective | Success Metric (30‑day) | Success Metric (90‑day) |
|-----------|------------------------|--------------------------|
| **Awareness** | 2 000 unique visitors to the repo page, 5 000 impressions on social posts | 10 000 unique visitors, 25 000 impressions |
| **Community** | 30 star ⭐, 5 forks, 2 external PRs | 100 star, 25 forks, 10 external PRs |
| **Adoption** | 10 fresh‑clone “quick‑start” runs (tracked via GitHub Actions badge) | 30 runs, 5 organizations reporting production use (via issue template) |
| **Lead Generation** | 20 newsletter sign‑ups / contact forms from the repo landing page | 50 sign‑ups, 3 demo‑request meetings |
| **Revenue (optional)** | 1 paid consulting/implementation inquiry | 3 inquiries, $5k+ pipeline |

---

## 2️⃣ Target Audiences (r

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 99b78403-fe5f-4adc-8c3f-fb44a2a54887                                                                       │
│  Final Output: # 📈 Growth & Launch Strategy for **“E‑Commerce Microservices Architecture on AWS EKS”**         │
│  *(GitHub repo: https://github.com/yourorg/ecommerce-eks)*                                                      │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## 1️⃣ Strategy Overview                                                                                        │
│                                                                                                                 │
│  | Objective | Success Metric (30‑day) | Success Metric (90‑day) |                                              │
│  |-----------|------------------------|--------------------------|                                              │
│  | **Awareness** | 2 000 unique visitors to the repo page, 5 000 impressions on social posts | 10 000 unique    │
│  visitors, 25 000 impressions |                                                                                 │
│  | **Community** | 30 star ⭐, 5 forks, 2 external PRs | 100 star, 25 forks, 10 external PRs |                  │
│  | **Adoption** | 10 fresh‑clone “quick‑start” runs (tracked via GitHub Actions badge) | 30 runs, 5             │
│  organizations reporting production use (via issue template) |                                                  │
│  | **Lead Generation** | 20 newsletter sign‑ups / contact forms from the repo landing page | 50 sign‑ups, 3     │
│  demo‑request meetings |                                                                                        │
│  | **Revenue (optional)** | 1 paid consulting/implementation inquiry | 3 inquiries, $5k+ pipeline |             │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## 2️⃣ Target Audiences (recapped)                                                                              │
│                                                                                                                 │
│  | Segment | Primary Pain Point | How the Repo Solves It |                                                      │
│  |---------|-------------------|------------------------|                                                       │
│  | **DevOps engineers / Platform teams** | Need a production‑grade, end‑to‑end blueprint (IaC + CI/CD +         │
│  Observability) | Full Terraform + Jenkins + Prometheus stack, ready to fork |                                  │
│  | **Students & AWS Academy participants** | Limited IAM permissions, cost‑concerns | Works with `LabRole`,     │
│  includes cost warnings & destroy script |                                                                      │
│  | **Software architects** | Want a reference for micro‑service design on EKS | HA‑ready architecture, RBAC,    │
│  multi‑pod Kaniko |                                   

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯